# ENN583 — Week 13 Practical, Part 2: When should the robot trust its model?

A classifier can be accurate on average and still fail dangerously. In this notebook you will audit a warehouse route classifier, deliberately shift its inputs, and design a policy that can abstain. The task is binary: predict whether the robot's immediate route is **blocked** or **free**. Each split contains only 70 examples, which is enough for a compact deployment case study but not for precise confidence-bin calibration estimates.

By the end you should be able to: test corruption severity; recognise confidently wrong predictions; interpret a risk–coverage curve; compare simple out-of-distribution (OOD) scores; estimate uncertainty with MC Dropout; and state an appropriate fallback. Reliability diagrams, ECE, and temperature scaling require larger evaluation sets and are covered in Part 1.

## 0. Setup

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageFilter
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from transformers import AutoImageProcessor, AutoModel

# Fix random seeds so the prediction head starts from the same weights.
SEED = 583
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# Support launching Jupyter from the repository root or this notebook's folder.
week8_candidates = [Path('../Week_08'), Path('practicals/Week_08')]
WEEK8_DIR = next((p for p in week8_candidates if p.exists()), week8_candidates[0])
DATA_DIR = WEEK8_DIR / 'warehouse_dataset'
ZIP_PATH = WEEK8_DIR / 'warehouse_dataset_v1.zip'
DATASET_URL = ('https://github.com/nikosuenderhauf/enn583/releases/download/'
               '2026-week-08-dataset-v1/warehouse_dataset_v1.zip')

# Download and unpack only when the prepared dataset is not already present.
if not DATA_DIR.exists():
    WEEK8_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading the warehouse dataset...')
    urlretrieve(DATASET_URL, ZIP_PATH)
    with ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_DIR)
print('Dataset:', DATA_DIR)

# ImageFolder assigns integer labels from the class-directory names.
raw_datasets = {s: ImageFolder(DATA_DIR / s) for s in ('train', 'val', 'test')}
class_to_idx = raw_datasets['train'].class_to_idx
assert class_to_idx == {'blocked': 0, 'free': 1}
print({s: len(ds) for s, ds in raw_datasets.items()}, class_to_idx)

## 1. Build the warehouse route classifier

We use a frozen DINOv2 vision backbone to convert each image into a 384-value feature vector, then train a small binary prediction head. Freezing the backbone makes the exercise fast and repeatable. The head includes dropout: dropout is disabled for ordinary predictions and deliberately re-enabled for the uncertainty experiment near the end.

In [ ]:
MODEL_ID = 'facebook/dinov2-small'
# The processor applies the resize and normalisation expected by DINOv2.
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
backbone = AutoModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
# Freeze all backbone weights: only the small head below will be trained.
for parameter in backbone.parameters():
    parameter.requires_grad = False

# Convert a list of PIL images into one batched pixel tensor.
def collate_images(batch):
    images, targets = zip(*batch)
    pixels = processor(images=list(images), return_tensors='pt')['pixel_values']
    return pixels, torch.tensor(targets)

def extract_features_from_images(images, batch_size=8):
    batches = []
    with torch.inference_mode():
        for start in range(0, len(images), batch_size):
            pixels = processor(images=images[start:start + batch_size], return_tensors='pt')['pixel_values']
            output = backbone(pixel_values=pixels.to(DEVICE))
            # Token 0 is DINOv2's CLS token: one 384-value vector per image.
            batches.append(output.last_hidden_state[:, 0].cpu())
    return torch.cat(batches)

def extract_split(dataset):
    loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_images)
    xs, ys = [], []
    with torch.inference_mode():
        for pixels, targets in loader:
            output = backbone(pixel_values=pixels.to(DEVICE))
            # Move cached features back to CPU to release accelerator memory.
            xs.append(output.last_hidden_state[:, 0].cpu())
            ys.append(targets)
    return torch.cat(xs), torch.cat(ys)

features, labels = {}, {}
for split in ('train', 'val', 'test'):
    features[split], labels[split] = extract_split(raw_datasets[split])
    print(split, tuple(features[split].shape))

In [ ]:
# Only this small head is trained; the cached DINOv2 features stay fixed.
head = nn.Sequential(
    nn.Linear(features['train'].shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.25),  # randomly hides 25% of hidden activations during training
    nn.Linear(128, len(class_to_idx)),
)
optimiser = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

# The dataset is small enough to update from all cached training features
# in one batch; no image processing occurs inside this training loop.
for epoch in range(400):
    head.train()
    optimiser.zero_grad()
    loss = loss_fn(head(features['train']), labels['train'])
    loss.backward()  # compute gradients for the head parameters
    optimiser.step()

# eval() disables dropout, making ordinary predictions deterministic.
head.eval()
with torch.inference_mode():
    logits = {s: head(features[s]) for s in ('val', 'test')}
for split in ('val', 'test'):
    accuracy = (logits[split].argmax(1) == labels[split]).float().mean()
    print(f'{split} accuracy: {accuracy:.3f}')

## 2. Controlled distribution shift

The clean accuracy above tells us how the classifier performs when evaluation images resemble its training data. A robot will also encounter degraded images: dim lighting, motion blur, dirty lenses, vibration, and other conditions that may not be represented in a clean test set. High clean accuracy therefore does not tell us how gracefully the system fails.

We now alter the test images without changing their labels. This is a controlled **covariate shift**: image appearance changes while the meaning of `blocked` and `free` remains fixed. Increasing severity combines darkness and blur to imitate degraded camera evidence. At each severity we compare accuracy with mean softmax confidence. If accuracy falls while confidence remains high, the classifier is failing silently rather than signalling that its evidence has become unreliable. These corruptions are illustrative, not a complete model of warehouse deployment.

In [ ]:
def corrupt(image, severity):
    # Increasing severity simultaneously darkens and blurs the camera image.
    image = ImageEnhance.Brightness(image).enhance(1.0 - 0.14 * severity)
    return image.filter(ImageFilter.GaussianBlur(radius=0.65 * severity))

# Retrieve the original PIL images so each severity starts from clean data.
test_images = [raw_datasets['test'][i][0] for i in range(len(raw_datasets['test']))]
severity_results = []
shifted_logits = {}
shifted_features = {}  # retained for risk and MC Dropout experiments
shifted_images = {}    # retained so uncertain examples can be displayed
for severity in range(6):
    images = [corrupt(im, severity) for im in test_images]
    # Re-extract visual features because corruption changes image content.
    current_features = extract_features_from_images(images)
    head.eval()
    with torch.inference_mode():
        current_logits = head(current_features)
    shifted_features[severity] = current_features
    shifted_images[severity] = images
    shifted_logits[severity] = current_logits
    # Softmax converts the two logits into blocked/free pseudo-probabilities.
    probs = current_logits.softmax(1)
    accuracy = current_logits.argmax(1).eq(labels['test']).float().mean().item()
    confidence = probs.max(1).values.mean().item()
    # This aggregate gap is descriptive, not ECE: a positive value means
    # mean confidence exceeds overall accuracy at this severity.
    confidence_gap = confidence - accuracy
    severity_results.append((severity, accuracy, confidence, confidence_gap))
    print(f'severity={severity}: accuracy={accuracy:.3f}, mean confidence={confidence:.3f}, gap={confidence_gap:+.3f}')

In [ ]:
severity, accuracy, confidence, confidence_gap = map(np.array, zip(*severity_results))
fig, axis = plt.subplots(figsize=(7, 4))
axis.plot(severity, accuracy, 'o-', label='accuracy')
axis.plot(severity, confidence, 'o-', label='mean confidence')
# Shade only regions where aggregate confidence is higher than accuracy.
axis.fill_between(severity, accuracy, confidence, where=confidence >= accuracy,
                  alpha=.18, color='tab:red', label='overconfidence gap')
axis.set(xlabel='Corruption severity', ylabel='Value', ylim=(0, 1.02))
axis.legend(); axis.grid(alpha=.3)
plt.show()

**Interpret the result:** Start at severity zero and follow both curves. Look for the point where accuracy begins to fall while mean confidence remains close to one. The shaded gap is not ECE; it is a simple aggregate warning that the model is more confident than its overall success rate justifies.

**Discuss:** At which severity does confidence become misleading? Would a fixed confidence alarm detect these failures? Why is corruption evaluation different from merely adding the same corruptions to the training set? What other camera failures should a warehouse robot be tested against?

## 3. Selective prediction and risk–coverage

The corruption experiment showed that the model can remain confident as its accuracy falls. A natural response is to let the classifier answer only when its confidence is sufficiently high and **abstain** otherwise. This helps only if confidence ranks correct predictions ahead of errors; a confidently wrong prediction will still pass the threshold.

**Coverage** is the fraction of inputs answered, while **selective risk** is the error rate among those answered inputs. At 100% coverage, selective risk is simply the ordinary error rate. Raising the confidence threshold rejects more inputs and lowers coverage. If confidence is useful, risk should fall as we move toward lower coverage. We compare the clean and severely corrupted curves to test whether this ranking survives distribution shift.

Abstention transfers responsibility to a fallback; it does not make the system safe by itself. Stopping, requesting another camera view, using another sensor, slowing down, or asking a human all have their own costs and failure modes.

In [ ]:
def risk_coverage(logits, targets):
    probabilities = logits.softmax(1)
    confidence, prediction = probabilities.max(1)
    # Sort once, then evaluate every possible prefix of accepted predictions.
    order = confidence.argsort(descending=True)
    # Convert correctness into 0/1 errors in confidence-ranked order.
    errors = prediction[order].ne(targets[order]).float()
    answered = torch.arange(1, len(errors) + 1)
    coverage = answered.float() / len(errors)
    # Each prefix represents accepting that many most-confident predictions.
    risk = errors.cumsum(0) / answered
    return coverage.numpy(), risk.numpy(), confidence[order].numpy()

fig, axis = plt.subplots(figsize=(6, 4))
for name, current in [('clean', logits['test']), ('severity 5', shifted_logits[5])]:
    coverage, risk, ordered_confidence = risk_coverage(current, labels['test'])
    axis.plot(coverage, risk, label=name)
axis.set(xlabel='Coverage (fraction answered)', ylabel='Selective risk (error among answered)', xlim=(0, 1), ylim=(0, None))
axis.grid(alpha=.3); axis.legend(); plt.show()

**Your turn:** use the validation set to choose the largest coverage whose selective risk is at most 5%. Report the implied confidence threshold, then apply that fixed threshold to the clean and severity-5 test sets. Never choose an operating threshold on the test set. Compare the right endpoints first, then compare risk at similar coverage. Did the validation behaviour transfer? If it did not, explain why a threshold selected on clean data is not a deployment guarantee.

In [ ]:
target_risk = 0.05
# Select the operating point using held-out validation data only.
coverage, risk, ordered_confidence = risk_coverage(logits['val'], labels['val'])
# Find every validation prefix satisfying the risk requirement.
valid = np.where(risk <= target_risk)[0]
if len(valid):
    # The final valid prefix gives the greatest supported coverage.
    selected = valid[-1]
    abstention_threshold = ordered_confidence[selected]
    print(f'Validation coverage={coverage[selected]:.3f}, risk={risk[selected]:.3f}, threshold={abstention_threshold:.3f}')
else:
    abstention_threshold = 1.0
    print('No non-empty validation operating point meets the risk target.')

def evaluate_abstention(current_logits, targets, threshold):
    probability = current_logits.softmax(1)
    confidence, prediction = probability.max(1)
    # Predictions below threshold are abstentions and excluded from risk.
    answered = confidence >= threshold
    coverage = answered.float().mean().item()
    risk = prediction[answered].ne(targets[answered]).float().mean().item() if answered.any() else float('nan')
    return coverage, risk

for name, current in [('clean test', logits['test']), ('severity-5 test', shifted_logits[5])]:
    test_coverage, test_risk = evaluate_abstention(current, labels['test'], abstention_threshold)
    print(f'{name:15s}: coverage={test_coverage:.3f}, selective risk={test_risk:.3f}')

## 4. Can confidence detect out-of-distribution inputs?

The fixed abstention threshold above may fail because confidence can remain high under corruption. Could we instead detect inputs that differ from the supported warehouse data? This is an **out-of-distribution (OOD) detection** question, which is different from asking whether a prediction is correct: an in-distribution image can be misclassified, and an OOD image can accidentally receive the right label.

We use unrelated object photographs bundled with the course as a small, obvious OOD set. **Maximum softmax probability (MSP)** is the largest class probability; low MSP may indicate uncertainty. **Predictive entropy** is high when probability is spread across classes. Neither score knows what is truly outside the training distribution, and a neural network can be highly confident on an unfamiliar image because softmax must distribute all probability across `blocked` and `free`. A serious evaluation would include several deployment-relevant near-OOD sets—such as a new warehouse, camera, or obstacle type—as well as obvious far-OOD examples.

In [ ]:
week10_candidates = [Path('../Week_10/images'), Path('practicals/Week_10/images')]
WEEK10_IMAGES = next((p for p in week10_candidates if p.exists()), week10_candidates[0])
# Sorting makes the selected OOD examples repeatable across operating systems.
ood_paths = sorted([p for p in WEEK10_IMAGES.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])[:40]
assert ood_paths, f'No OOD images found under {WEEK10_IMAGES}'
ood_images = [Image.open(path).convert('RGB') for path in ood_paths]
ood_features = extract_features_from_images(ood_images)
head.eval()
with torch.inference_mode():
    # The classifier must still choose between blocked and free, even though
    # these images do not belong to the supported warehouse task.
    ood_logits = head(ood_features)

def msp_and_entropy(current_logits):
    p = current_logits.softmax(1)
    # Binary entropy is lowest near probability 0 or 1 and highest near 0.5.
    # clamp_min avoids log(0) without materially changing probabilities.
    return p.max(1).values.numpy(), (-(p * p.clamp_min(1e-12).log()).sum(1)).numpy()

id_msp, id_entropy = msp_and_entropy(logits['test'])
ood_msp, ood_entropy = msp_and_entropy(ood_logits)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Density normalisation allows sets with different image counts to be compared.
axes[0].hist(id_msp, bins=12, alpha=.6, label='warehouse test', density=True)
axes[0].hist(ood_msp, bins=12, alpha=.6, label='OOD', density=True)
axes[0].set(xlabel='Maximum softmax probability', ylabel='Density'); axes[0].legend()
axes[1].hist(id_entropy, bins=12, alpha=.6, label='warehouse test', density=True)
axes[1].hist(ood_entropy, bins=12, alpha=.6, label='OOD', density=True)
axes[1].set(xlabel='Predictive entropy', ylabel='Density'); axes[1].legend()
plt.show()

**Interpret the result:** Perfect OOD separation would place the warehouse and OOD histograms in distinct regions. Overlap means that no single threshold can reject every OOD input without also rejecting supported warehouse images.

**Discuss:** Do the score distributions overlap? Find an OOD image with high confidence. Why can a two-class softmax be confident about an image belonging to neither class? Why would reporting one obvious OOD dataset be insufficient evidence for deployment?

## 5. MC Dropout: does repeated inference reveal uncertainty?

The MSP and entropy histograms above use only one deterministic forward pass. Their overlap shows that output probabilities alone may not reveal when the model lacks knowledge. We will now ask a different question: does the prediction change when we perturb the model itself? If several plausible versions of the prediction head disagree, that disagreement may indicate limited knowledge about the input.

Ordinary `eval()` inference disables dropout and returns the same head output every time. For **MC Dropout**, we deliberately keep dropout active and run the head many times. Each pass removes a different random subset of hidden activations, producing a sample of plausible predictions from this one trained head. We compare clean images, severely corrupted images, and unrelated OOD images to see whether stochastic disagreement rises as inputs move away from the supported clean data.

We report two quantities. **Predictive entropy** measures ambiguity in the mean prediction. **Mutual information (MI)** measures disagreement across stochastic passes and is often used as an approximate epistemic-uncertainty score. Here dropout exists only in the small prediction head; the frozen DINOv2 backbone remains deterministic. Therefore this experiment measures sensitivity of the head, not total uncertainty in the complete vision system.

In [ ]:
def mc_dropout(input_features, passes=50):
    # train() activates the Dropout layer. The frozen backbone is separate
    # and remains in eval mode, so only the prediction head is stochastic.
    head.train()
    with torch.inference_mode():
        samples = torch.stack([
            head(input_features).softmax(1) for _ in range(passes)
        ])  # shape: [passes, images, classes]
    head.eval()  # restore deterministic behaviour for later use
    mean_probability = samples.mean(0)
    # Entropy of the average prediction: high when the final answer is ambiguous.
    predictive_entropy = -(mean_probability * mean_probability.clamp_min(1e-12).log()).sum(1)
    # Average entropy within individual passes. Subtracting it from predictive
    # entropy isolates disagreement between passes (mutual information).
    expected_entropy = -(samples * samples.clamp_min(1e-12).log()).sum(2).mean(0)
    mutual_information = predictive_entropy - expected_entropy
    return mean_probability, predictive_entropy, mutual_information

mc_inputs = {
    'clean warehouse': features['test'],
    'severity-5 corruption': shifted_features[5],
    'unrelated OOD': ood_features,
}
# Run the identical stochastic procedure for all three input groups.
mc_results = {name: mc_dropout(values) for name, values in mc_inputs.items()}

for name, (_, entropy, mi) in mc_results.items():
    print(f'{name:24s}: mean entropy={entropy.mean():.4f}, mean MI={mi.mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = list(mc_results)
# Box plots expose overlap and outliers that a single mean would hide.
axes[0].boxplot([mc_results[name][1].numpy() for name in names], labels=names)
axes[0].set(ylabel='Predictive entropy', title='Ambiguity of mean prediction')
axes[1].boxplot([mc_results[name][2].numpy() for name in names], labels=names)
axes[1].set(ylabel='Mutual information', title='Disagreement across dropout passes')
for axis in axes:
    axis.tick_params(axis='x', rotation=20)
    axis.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

**Interpret the result:** MC Dropout is operational here: the code performs 50 stochastic forward passes, restores deterministic evaluation afterwards, and compares complete score distributions for clean, corrupted, and OOD inputs. A useful signal would tend to be larger on corrupted or OOD inputs, but overlap is expected. Low MI does **not** prove that an image is safe or familiar; all dropout passes can agree on the same wrong answer. Results also depend on where dropout was placed and how the head was trained.

**Your turn:** Which input group has the largest typical entropy and MI? Do the box plots separate the groups well enough to choose a reliable threshold? Find at least one unfamiliar or corrupted example whose MI remains low, and explain why MC Dropout should be combined with task checks, monitoring, and a fallback rather than used as a safety guarantee.

## 6. Bring the evidence together: failure audit and deployment card

No single method above solved the trust problem. Confidence remained high under corruption, a clean-data abstention threshold did not preserve its risk under shift, OOD-score distributions overlapped, and MC Dropout provided only a noisy additional signal. A responsible deployment decision must therefore combine quantitative evidence with monitoring, fallback behaviour, and an explicit statement of what has not been tested.

Bring your results together in a short deployment card containing:

1. **Intended use:** the environment, camera and decision supported by the model.
2. **Evidence:** clean accuracy, corruption results, the confidence–accuracy gap, and a risk–coverage operating point.
3. **Silent failures:** at least two confidently wrong or unsupported examples.
4. **Monitoring:** what input, score, temporal or system signals should be logged.
5. **Fallback:** what the robot should do when the model abstains.
6. **Residual risk:** one important failure not addressed by abstention, OOD scoring, or MC Dropout.

A good mitigation reduces a stated risk under stated conditions. It does not prove that the robot is safe.